# Paper-Like Multimodal Contact Classification

Self-contained Kaggle notebook. It trains three models separately:

- audio-only: AST + CLAP + Transformer fusion
- video/image-only: ViT-B/16
- fusion: AST + CLAP + ViT-B/16 + Transformer fusion

Outputs after running all training cells:

- `best_audio_model.pt`, `audio_results.json`
- `best_video_model.pt`, `video_results.json`
- `best_fusion_model.pt`, `fusion_results.json`

## 1. Setup

Set `DATA_ROOT` to the folder containing `audio_visual_dataset_default/` and `audio_visual_dataset_robo_default/`. On Kaggle, change the first path if your dataset name is different.

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/kaggle/input/contact-data/dataset')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input/contact_data/dataset')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('../dataset')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('dataset')

OUTPUT_DIR = Path('/kaggle/working/outputs') if Path('/kaggle/working').exists() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT =', DATA_ROOT.resolve())
print('OUTPUT_DIR =', OUTPUT_DIR.resolve())

If Kaggle does not include `transformers`, uncomment the install cell. Pretrained AST/CLAP/ViT weights require Kaggle internet or an attached cache/model dataset.

In [ ]:
# !pip install -q transformers

## 2. Imports and Config

In [ ]:
import json
import math
import random
import wave
from dataclasses import asdict, dataclass
from typing import Optional

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm.auto import tqdm

LABELS = ('ambient', 'leaf', 'trunk', 'twig')
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
SPLITS = ('audio_visual_dataset_default', 'audio_visual_dataset_robo_default')

@dataclass
class DataConfig:
    data_root: Path = DATA_ROOT
    target_sample_rate: int = 22000
    audio_window_sec: float = 0.8
    image_size: int = 224
    skip_missing_files: bool = True
    train_crop: str = 'random'
    eval_crop: str = 'center'

@dataclass
class TrainConfig:
    epochs: int = 5
    batch_size: int = 4
    lr: float = 1e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    val_size: float = 0.2
    seed: int = 42
    use_amp: bool = True

@dataclass
class ModelConfig:
    ast_model_name: str = 'MIT/ast-finetuned-audioset-10-10-0.4593'
    clap_model_name: str = 'laion/clap-htsat-unfused'
    fusion_dim: int = 256
    fusion_heads: int = 4
    fusion_layers: int = 2
    fusion_dropout: float = 0.1
    freeze_pretrained: bool = False

data_cfg = DataConfig()
train_cfg = TrainConfig()
model_cfg = ModelConfig()

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(train_cfg.seed)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)

## 3. Dataset Index and Metrics

In [ ]:
def build_index(data_root: Path, skip_missing_files=True):
    rows, skipped = [], 0
    for split_name in SPLITS:
        split_dir = data_root / split_name
        csv_path = split_dir / 'dataset.csv'
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        for item in df.to_dict('records'):
            audio_path = split_dir / item['audio_file']
            image_path = split_dir / item['image_file']
            exists = audio_path.exists() and image_path.exists()
            if skip_missing_files and not exists:
                skipped += 1
                continue
            rows.append({
                'split_name': split_name,
                'audio_path': str(audio_path),
                'image_path': str(image_path),
                'label': item['category'],
                'label_id': LABEL_TO_ID[item['category']],
                'files_exist': exists,
            })
    index = pd.DataFrame(rows)
    index.attrs['skipped_missing_files'] = skipped
    return index

def make_train_val_split(df, val_size=0.2, seed=42):
    try:
        from sklearn.model_selection import train_test_split
        train_df, val_df = train_test_split(df, test_size=val_size, random_state=seed, stratify=df['label_id'])
    except Exception:
        train_df = df.sample(frac=1.0 - val_size, random_state=seed)
        val_df = df.drop(train_df.index)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)

def f1_scores(y_true, y_pred, num_classes=4):
    try:
        from sklearn.metrics import f1_score
        return {
            'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        }
    except Exception:
        y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
        per_class, supports = [], []
        for cls in range(num_classes):
            true_cls, pred_cls = y_true == cls, y_pred == cls
            tp = float(np.logical_and(true_cls, pred_cls).sum())
            fp = float(np.logical_and(~true_cls, pred_cls).sum())
            fn = float(np.logical_and(true_cls, ~pred_cls).sum())
            denom = 2 * tp + fp + fn
            per_class.append(0.0 if denom == 0 else 2 * tp / denom)
            supports.append(float(true_cls.sum()))
        per_class, supports = np.asarray(per_class), np.asarray(supports)
        weighted = 0.0 if supports.sum() == 0 else float((per_class * supports).sum() / supports.sum())
        return {'macro_f1': float(per_class.mean()), 'weighted_f1': weighted}

def binary_contact_f1(y_true, y_pred):
    ambient_id = LABEL_TO_ID['ambient']
    true_contact = np.asarray(y_true) != ambient_id
    pred_contact = np.asarray(y_pred) != ambient_id
    try:
        from sklearn.metrics import f1_score
        return float(f1_score(true_contact, pred_contact, zero_division=0))
    except Exception:
        tp = float(np.logical_and(true_contact, pred_contact).sum())
        fp = float(np.logical_and(~true_contact, pred_contact).sum())
        fn = float(np.logical_and(true_contact, ~pred_contact).sum())
        denom = 2 * tp + fp + fn
        return 0.0 if denom == 0 else 2 * tp / denom

def compute_metrics(logits_list, labels_list):
    logits = torch.cat(logits_list, dim=0)
    labels = torch.cat(labels_list, dim=0)
    y_true = labels.numpy()
    y_pred = logits.argmax(dim=1).numpy()
    out = f1_scores(y_true, y_pred, num_classes=len(LABELS))
    out['binary_contact_f1'] = binary_contact_f1(y_true, y_pred)
    out['accuracy'] = float((y_true == y_pred).mean())
    return out

## 4. Audio Loading and Dataset

In [ ]:
class AudioPipeline:
    def __init__(self, target_sample_rate=22000, window_sec=0.8):
        try:
            import torchaudio
        except (ImportError, OSError):
            torchaudio = None
        self.torchaudio = torchaudio
        self.target_sample_rate = target_sample_rate
        self.window_samples = int(round(target_sample_rate * window_sec))
        self._resamplers = {}

    def load_waveform(self, path):
        if self.torchaudio is not None:
            waveform, sample_rate = self.torchaudio.load(path)
        else:
            waveform, sample_rate = self._load_wav_stdlib(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sample_rate != self.target_sample_rate:
            waveform = self._resample(waveform, sample_rate)
        return waveform

    def _resample(self, waveform, sample_rate):
        if self.torchaudio is not None:
            if sample_rate not in self._resamplers:
                self._resamplers[sample_rate] = self.torchaudio.transforms.Resample(
                    orig_freq=sample_rate,
                    new_freq=self.target_sample_rate,
                    lowpass_filter_width=64,
                    rolloff=0.9475937167399596,
                    resampling_method='sinc_interp_kaiser',
                )
            return self._resamplers[sample_rate](waveform)
        from scipy.signal import resample_poly
        gcd = math.gcd(sample_rate, self.target_sample_rate)
        up = self.target_sample_rate // gcd
        down = sample_rate // gcd
        out = resample_poly(waveform.numpy(), up=up, down=down, axis=-1)
        return torch.from_numpy(out.copy()).float()

    def crop_or_pad(self, waveform, mode='center'):
        total, target = waveform.shape[-1], self.window_samples
        if total < target:
            return F.pad(waveform, (0, target - total))
        if total == target:
            return waveform
        max_start = total - target
        if mode == 'random':
            start = int(torch.randint(0, max_start + 1, (1,)).item())
        elif mode == 'energy':
            energy = waveform.pow(2).mean(dim=0, keepdim=True).unsqueeze(0)
            kernel = torch.ones(1, 1, target)
            start = int(F.conv1d(energy, kernel).argmax(dim=-1).item())
        else:
            start = max_start // 2
        return waveform[..., start:start + target]

    def __call__(self, path, crop_mode='center'):
        waveform = self.load_waveform(path)
        waveform = self.crop_or_pad(waveform, crop_mode)
        return waveform.squeeze(0)

    @staticmethod
    def _load_wav_stdlib(path):
        with wave.open(path, 'rb') as wav:
            channels = wav.getnchannels()
            sample_width = wav.getsampwidth()
            sample_rate = wav.getframerate()
            frames = wav.readframes(wav.getnframes())
        if sample_width == 2:
            data = torch.frombuffer(bytearray(frames), dtype=torch.int16).float() / float(2**15)
        elif sample_width == 1:
            data = (torch.frombuffer(bytearray(frames), dtype=torch.uint8).float() - 128.0) / 255.0
        else:
            raise ValueError(f'Unsupported WAV sample width: {sample_width}')
        return data.view(-1, channels).t().contiguous(), sample_rate

class ContactDataset(Dataset):
    def __init__(self, frame, config: DataConfig, mode='fusion', train=False):
        self.frame = frame.reset_index(drop=True)
        self.config = config
        self.mode = mode
        self.train = train
        self.crop_mode = config.train_crop if train else config.eval_crop
        self.audio = AudioPipeline(config.target_sample_rate, config.audio_window_sec)
        self.image_transform = transforms.Compose([
            transforms.Resize((config.image_size, config.image_size)),
            transforms.RandomHorizontalFlip(p=0.5 if train else 0.0),
            transforms.ToTensor(),
            transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ])

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        out = {'label': torch.tensor(row.label_id, dtype=torch.long)}
        if self.mode in ('audio', 'fusion'):
            out['waveform'] = self.audio(row.audio_path, self.crop_mode)
        if self.mode in ('video', 'fusion'):
            image = Image.open(row.image_path).convert('RGB')
            out['image'] = self.image_transform(image)
        return out

## 5. Inspect Dataset

In [ ]:
index = build_index(data_cfg.data_root, data_cfg.skip_missing_files)
print('Valid samples:', len(index))
print('Skipped missing files:', index.attrs.get('skipped_missing_files', 0))
display(index['split_name'].value_counts())
display(index['label'].value_counts().reindex(LABELS).fillna(0).astype(int))

train_df, val_df = make_train_val_split(index, train_cfg.val_size, train_cfg.seed)
print('Train:', len(train_df), 'Val:', len(val_df))

## 6. Models

In [ ]:
class TransformerFusionHead(nn.Module):
    def __init__(self, input_dims, num_classes=4, fusion_dim=256, heads=4, layers=2, dropout=0.1):
        super().__init__()
        self.projections = nn.ModuleList([nn.Linear(dim, fusion_dim) for dim in input_dims])
        self.cls_token = nn.Parameter(torch.zeros(1, 1, fusion_dim))
        layer = nn.TransformerEncoderLayer(
            d_model=fusion_dim,
            nhead=heads,
            dim_feedforward=fusion_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers)
        self.classifier = nn.Sequential(nn.LayerNorm(fusion_dim), nn.Linear(fusion_dim, num_classes))

    def forward(self, embeddings):
        tokens = torch.stack([proj(emb) for proj, emb in zip(self.projections, embeddings)], dim=1)
        cls = self.cls_token.expand(tokens.size(0), -1, -1)
        fused = self.encoder(torch.cat([cls, tokens], dim=1))
        return self.classifier(fused[:, 0])

class ASTCLAPAudioModel(nn.Module):
    def __init__(self, cfg: ModelConfig, sample_rate=22000, num_classes=4):
        super().__init__()
        from transformers import ASTFeatureExtractor, ASTModel, AutoProcessor, ClapModel
        self.sample_rate = sample_rate
        self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained(cfg.ast_model_name)
        self.ast_model = ASTModel.from_pretrained(cfg.ast_model_name)
        self.clap_processor = AutoProcessor.from_pretrained(cfg.clap_model_name)
        self.clap_model = ClapModel.from_pretrained(cfg.clap_model_name)
        self.ast_sample_rate = getattr(self.ast_feature_extractor, 'sampling_rate', sample_rate)
        self.clap_sample_rate = getattr(getattr(self.clap_processor, 'feature_extractor', None), 'sampling_rate', sample_rate)
        ast_dim = self.ast_model.config.hidden_size
        clap_dim = self.clap_model.config.projection_dim
        self.head = TransformerFusionHead(
            [ast_dim, clap_dim], num_classes, cfg.fusion_dim, cfg.fusion_heads, cfg.fusion_layers, cfg.fusion_dropout
        )
        if cfg.freeze_pretrained:
            self.freeze_backbones()

    def freeze_backbones(self):
        for module in (self.ast_model, self.clap_model):
            for param in module.parameters():
                param.requires_grad = False

    def encode_ast(self, waveform):
        arrays = self.to_processor_arrays(waveform, self.ast_sample_rate)
        inputs = self.ast_feature_extractor(arrays, sampling_rate=self.ast_sample_rate, return_tensors='pt', padding=True)
        inputs = {k: v.to(waveform.device) for k, v in inputs.items()}
        outputs = self.ast_model(**inputs)
        return outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]

    def encode_clap(self, waveform):
        arrays = self.to_processor_arrays(waveform, self.clap_sample_rate)
        inputs = self.clap_processor(audio=arrays, sampling_rate=self.clap_sample_rate, return_tensors='pt', padding=True)
        inputs = {k: v.to(waveform.device) for k, v in inputs.items()}
        return self.clap_model.get_audio_features(**inputs)

    def to_processor_arrays(self, waveform, target_rate):
        arrays = [item.detach().float().cpu().numpy() for item in waveform]
        if target_rate == self.sample_rate:
            return arrays
        from scipy.signal import resample_poly
        gcd = math.gcd(self.sample_rate, target_rate)
        up = target_rate // gcd
        down = self.sample_rate // gcd
        return [resample_poly(item, up=up, down=down).astype('float32') for item in arrays]

    def forward(self, waveform=None, image=None):
        return self.head([self.encode_ast(waveform), self.encode_clap(waveform)])

class ViTVideoModel(nn.Module):
    def __init__(self, cfg: ModelConfig, num_classes=4):
        super().__init__()
        weights = models.ViT_B_16_Weights.DEFAULT
        self.vit = models.vit_b_16(weights=weights)
        vit_dim = self.vit.heads.head.in_features
        self.vit.heads = nn.Identity()
        self.classifier = nn.Sequential(nn.LayerNorm(vit_dim), nn.Linear(vit_dim, num_classes))
        if cfg.freeze_pretrained:
            for param in self.vit.parameters():
                param.requires_grad = False

    def forward(self, waveform=None, image=None):
        return self.classifier(self.vit(image))

class PaperFusionModel(nn.Module):
    def __init__(self, cfg: ModelConfig, sample_rate=22000, num_classes=4):
        super().__init__()
        from transformers import ASTFeatureExtractor, ASTModel, AutoProcessor, ClapModel
        self.sample_rate = sample_rate
        self.ast_feature_extractor = ASTFeatureExtractor.from_pretrained(cfg.ast_model_name)
        self.ast_model = ASTModel.from_pretrained(cfg.ast_model_name)
        self.clap_processor = AutoProcessor.from_pretrained(cfg.clap_model_name)
        self.clap_model = ClapModel.from_pretrained(cfg.clap_model_name)
        self.ast_sample_rate = getattr(self.ast_feature_extractor, 'sampling_rate', sample_rate)
        self.clap_sample_rate = getattr(getattr(self.clap_processor, 'feature_extractor', None), 'sampling_rate', sample_rate)
        weights = models.ViT_B_16_Weights.DEFAULT
        self.vit = models.vit_b_16(weights=weights)
        vit_dim = self.vit.heads.head.in_features
        self.vit.heads = nn.Identity()
        ast_dim = self.ast_model.config.hidden_size
        clap_dim = self.clap_model.config.projection_dim
        self.head = TransformerFusionHead(
            [ast_dim, clap_dim, vit_dim], num_classes, cfg.fusion_dim, cfg.fusion_heads, cfg.fusion_layers, cfg.fusion_dropout
        )
        if cfg.freeze_pretrained:
            for module in (self.ast_model, self.clap_model, self.vit):
                for param in module.parameters():
                    param.requires_grad = False

    def encode_ast(self, waveform):
        arrays = self.to_processor_arrays(waveform, self.ast_sample_rate)
        inputs = self.ast_feature_extractor(arrays, sampling_rate=self.ast_sample_rate, return_tensors='pt', padding=True)
        inputs = {k: v.to(waveform.device) for k, v in inputs.items()}
        outputs = self.ast_model(**inputs)
        return outputs.pooler_output if outputs.pooler_output is not None else outputs.last_hidden_state[:, 0]

    def encode_clap(self, waveform):
        arrays = self.to_processor_arrays(waveform, self.clap_sample_rate)
        inputs = self.clap_processor(audio=arrays, sampling_rate=self.clap_sample_rate, return_tensors='pt', padding=True)
        inputs = {k: v.to(waveform.device) for k, v in inputs.items()}
        return self.clap_model.get_audio_features(**inputs)

    def to_processor_arrays(self, waveform, target_rate):
        arrays = [item.detach().float().cpu().numpy() for item in waveform]
        if target_rate == self.sample_rate:
            return arrays
        from scipy.signal import resample_poly
        gcd = math.gcd(self.sample_rate, target_rate)
        up = target_rate // gcd
        down = self.sample_rate // gcd
        return [resample_poly(item, up=up, down=down).astype('float32') for item in arrays]

    def forward(self, waveform=None, image=None):
        return self.head([self.encode_ast(waveform), self.encode_clap(waveform), self.vit(image)])

## 7. Training Utilities

In [ ]:
def make_loaders(mode):
    train_ds = ContactDataset(train_df, data_cfg, mode=mode, train=True)
    val_ds = ContactDataset(val_df, data_cfg, mode=mode, train=False)
    train_loader = DataLoader(
        train_ds, batch_size=train_cfg.batch_size, shuffle=True,
        num_workers=train_cfg.num_workers, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=train_cfg.batch_size, shuffle=False,
        num_workers=train_cfg.num_workers, pin_memory=True
    )
    return train_loader, val_loader

def build_model(mode):
    if mode == 'audio':
        return ASTCLAPAudioModel(model_cfg, data_cfg.target_sample_rate, len(LABELS))
    if mode == 'video':
        return ViTVideoModel(model_cfg, len(LABELS))
    if mode == 'fusion':
        return PaperFusionModel(model_cfg, data_cfg.target_sample_rate, len(LABELS))
    raise ValueError(f'Unknown mode: {mode}')

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    scaler = torch.amp.GradScaler('cuda', enabled=train_cfg.use_amp and device == 'cuda')
    total_loss, total_items = 0.0, 0
    logits_list, labels_list = [], []

    for batch in tqdm(loader, leave=False):
        labels = batch['label'].to(device)
        waveform = batch.get('waveform')
        image = batch.get('image')
        if waveform is not None:
            waveform = waveform.to(device)
        if image is not None:
            image = image.to(device)

        with torch.set_grad_enabled(is_train):
            with torch.amp.autocast('cuda', enabled=train_cfg.use_amp and device == 'cuda'):
                logits = model(waveform=waveform, image=image)
                loss = criterion(logits, labels)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * labels.size(0)
        total_items += labels.size(0)
        logits_list.append(logits.detach().cpu())
        labels_list.append(labels.detach().cpu())

    metrics = compute_metrics(logits_list, labels_list)
    metrics['loss'] = total_loss / total_items
    return metrics

def train_one_mode(mode):
    print(f'\n===== Training {mode} model =====')
    set_seed(train_cfg.seed)
    train_loader, val_loader = make_loaders(mode)
    model = build_model(mode).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=train_cfg.lr,
        weight_decay=train_cfg.weight_decay,
    )

    best_macro_f1 = -1.0
    history = []
    best_weight_path = OUTPUT_DIR / f'best_{mode}_model.pt'
    result_path = OUTPUT_DIR / f'{mode}_results.json'

    for epoch in range(1, train_cfg.epochs + 1):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer)
        val_metrics = run_epoch(model, val_loader, criterion, None)
        row = {'epoch': epoch, 'train': train_metrics, 'val': val_metrics}
        history.append(row)
        print(
            f"{mode} epoch={epoch} "
            f"train_loss={train_metrics['loss']:.4f} train_macro_f1={train_metrics['macro_f1']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} val_macro_f1={val_metrics['macro_f1']:.4f} "
            f"val_weighted_f1={val_metrics['weighted_f1']:.4f} val_binary_f1={val_metrics['binary_contact_f1']:.4f}"
        )
        if val_metrics['macro_f1'] > best_macro_f1:
            best_macro_f1 = val_metrics['macro_f1']
            torch.save({
                'mode': mode,
                'model': model.state_dict(),
                'labels': LABELS,
                'data_config': asdict(data_cfg),
                'train_config': asdict(train_cfg),
                'model_config': asdict(model_cfg),
                'best_val_metrics': val_metrics,
            }, best_weight_path)

    result = {
        'mode': mode,
        'best_macro_f1': best_macro_f1,
        'weight_file': str(best_weight_path),
        'history': history,
    }
    with open(result_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)
    print(f'Saved weight: {best_weight_path}')
    print(f'Saved result: {result_path}')
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 8. Train Audio-Only

Produces `best_audio_model.pt` and `audio_results.json`.

In [ ]:
audio_result = train_one_mode('audio')

## 9. Train Video/Image-Only

The released dataset stores one image frame per sample, so this branch is named `video` for reporting but uses ViT-B/16 over image frames. Produces `best_video_model.pt` and `video_results.json`.

In [ ]:
video_result = train_one_mode('video')

## 10. Train Fusion

Produces `best_fusion_model.pt` and `fusion_results.json`.

In [ ]:
fusion_result = train_one_mode('fusion')

## 11. Summary

In [ ]:
summary = pd.DataFrame([
    {'mode': r['mode'], 'best_macro_f1': r['best_macro_f1'], 'weight_file': r['weight_file']}
    for r in [audio_result, video_result, fusion_result]
])
display(summary)
print('Output files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print(path)